In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import List, Union
from datetime import datetime

# Input Models

class SafetyStatus(Enum):
    SAFE = "SAFE"
    BLOCKED = "BLOCKED"

@dataclass(frozen=True)
class SafetyStatusRecord:
    """Module 4 input."""
    status: SafetyStatus
    reason_codes: List[str]

@dataclass(frozen=True)
class CoolingRecommendationRecord:
    """Module 7 input."""
    recommended_reduction_percentage: float
    time_window: str  # Time window
    safety_status: str
    explanation: str

@dataclass(frozen=True)
class SavingsEstimationRecord:
    """Module 9 input."""
    kw_saved: float
    rupees_saved_per_interval: float

# Output Model

@dataclass(frozen=True)
class FinalRecommendationRecord:
    """Final recommendation."""
    action: str     # Action
    window: str     # Window
    savings: str    # Savings
    safety: str     # Safety
    reason: str     # Reason

# Module Implementation

class EnerviaRecommendationEngine:
    """Recommendation Generator."""

    def generate_final_recommendation(
        self,
        recommendation: CoolingRecommendationRecord,
        savings: SavingsEstimationRecord,
        safety_status: SafetyStatusRecord,
        context_reasons: List[str]
    ) -> FinalRecommendationRecord:
        """Generate final record."""
        
        # Check veto
        if safety_status.status == SafetyStatus.BLOCKED:
            return FinalRecommendationRecord(
                action="Maintain current settings (No Action)",
                window=recommendation.time_window,
                savings="0 kW | ₹0",
                safety="BLOCKED",
                reason=f"Safety Veto: {', '.join(safety_status.reason_codes)}"
            )

        # Format action
        if recommendation.recommended_reduction_percentage <= 0:
            action_str = "Maintain current settings"
            savings_str = "0 kW | ₹0"
            final_reason = "System already operating at peak efficiency."
        else:
            action_str = f"Reduce cooling by {recommendation.recommended_reduction_percentage}%"
            savings_str = f"{savings.kw_saved} kW | ₹{savings.rupees_saved_per_interval}"
            # Combine reasons
            final_reason = " + ".join(context_reasons)

        # Construct record
        return FinalRecommendationRecord(
            action=action_str,
            window=recommendation.time_window,
            savings=savings_str,
            safety="SAFE",
            reason=final_reason
        )

# engine = EnerviaRecommendationEngine()
# final_output = engine.generate_final_recommendation(
#     cooling_rec, savings_rec, safety_rec, reasons
# )